# Exploratory Data Analysis (EDA)

## WiDS Datathon 2026 - Wildfire Evacuation Zone Impact Prediction

This notebook performs comprehensive exploratory data analysis on the wildfire prediction dataset.

### 📋 Problem Statement

**Problem**: Predict the probability that a wildfire will threaten an evacuation zone (come within 5km) at multiple time horizons: 12h, 24h, 48h, and 72h after the initial 5-hour observation period.

**Why This Matters**: Early prediction of wildfire threats to evacuation zones can save lives by enabling timely evacuations and resource allocation.

### 🎯 Challenge Type

**Survival Analysis with Censored Data - Multi-Horizon Probability Forecasting**

**What is Survival Analysis?**
- Survival analysis models the time until an event occurs (in this case, when a fire reaches an evacuation zone)
- It handles "censored" data where the event hasn't occurred yet within our observation window
- Example: If a fire doesn't reach the zone within 72 hours, we don't know if it would have reached it at 80 hours - this is censored data

### 📊 Target Variables

- **`time_to_hit_hours`**: Time from t₀+5h until fire comes within 5km (0-72 hours)
  - This tells us *when* the fire reached the danger zone
  - Range: 0-72 hours after the initial 5-hour observation period
  
- **`event`**: Binary indicator (1 if fire hit within 72h, 0 if censored)
  - 1 = Fire reached the evacuation zone within 72 hours (event occurred)
  - 0 = Fire did NOT reach the zone within 72 hours (censored - we don't know what happened after)

### 📈 Evaluation Metric

**Hybrid Score = 0.3 × C-index + 0.7 × (1 - Weighted Brier Score)**

**Understanding the Metrics:**
- **C-index (Concordance Index)**: Measures how well the model ranks predictions (like AUC for survival analysis)
  - Higher is better (max = 1.0)
  - Checks if fires that hit sooner are predicted to have higher probabilities
  
- **Brier Score**: Measures the accuracy of probability predictions
  - Lower is better, so we use (1 - Brier Score) to make higher better
  - Weighted across multiple time horizons (12h, 24h, 48h, 72h)
  
- **Why Hybrid?**: Combines ranking quality (C-index) with probability calibration (Brier Score) for robust evaluation

## Setup and Configuration

### 🔧 Why This Setup Matters

Before diving into analysis, we need to:
1. **Import necessary libraries** for data manipulation and visualization
2. **Set random seeds** to ensure our analysis is reproducible (same results every time)
3. **Configure display settings** to see our data clearly
4. **Suppress warnings** to keep output clean (only for known, non-critical warnings)

In [ ]:
# ============================================================================
# LIBRARY IMPORTS
# ============================================================================

# Data manipulation and analysis
import pandas as pd  # For working with tabular data (DataFrames)
import numpy as np   # For numerical operations and array handling

# Visualization libraries
import matplotlib.pyplot as plt  # Core plotting library
import seaborn as sns            # Statistical visualization (built on matplotlib)

# Utilities
from pathlib import Path  # For handling file paths in a cross-platform way
import warnings

# Suppress warnings to keep output clean
# Note: Only do this after you've verified there are no critical warnings!
warnings.filterwarnings('ignore')

# ============================================================================
# REPRODUCIBILITY SETTINGS
# ============================================================================

# Set random seed for reproducibility
# Why? Many operations involve randomness (sampling, train/test splits, etc.)
# Setting a seed ensures we get the same "random" results every time we run the code
RANDOM_STATE = 42  # 42 is a common choice (Hitchhiker's Guide reference!)
np.random.seed(RANDOM_STATE)

# ============================================================================
# VISUALIZATION CONFIGURATION
# ============================================================================

# Set the visual style for all plots
plt.style.use('seaborn-v0_8-darkgrid')  # Professional-looking style with grid

# Set color palette for plots
# 'husl' provides distinct, colorblind-friendly colors
sns.set_palette("husl")

# Display plots inline in the notebook (Jupyter magic command)
%matplotlib inline

# ============================================================================
# PANDAS DISPLAY SETTINGS
# ============================================================================

# Show all columns when displaying DataFrames (no truncation with '...')
pd.set_option('display.max_columns', None)

# Show up to 100 rows (default is usually 60)
pd.set_option('display.max_rows', 100)

# Format floating point numbers to 4 decimal places for readability
# Example: 0.123456789 will display as 0.1235
pd.set_option('display.float_format', lambda x: f'{x:.4f}')

# ============================================================================
# CONFIRMATION
# ============================================================================

print("✅ Libraries imported successfully!")
print(f"✅ Random seed set to: {RANDOM_STATE}")
print(f"✅ Ready to begin exploratory data analysis!")

## 1. Load Data

### 📂 Understanding Our Datasets

We have three CSV files to load:

1. **`train.csv`**: Training data with features AND target variables
   - Used to train our model
   - Contains `time_to_hit_hours` and `event` columns

2. **`test.csv`**: Test data with features ONLY (no targets)
   - Used to make predictions for submission
   - We don't know the true outcomes for this data

3. **`metaData.csv`**: Information about each feature
   - Describes what each column represents
   - Groups features into categories (distance, growth, kinematics, etc.)
   - Helps us understand the domain context

In [ ]:
# ============================================================================
# LOAD DATASETS
# ============================================================================

# Load training data (includes target variables)
train_df = pd.read_csv('../data/train.csv')

# Load test data (features only, no targets)
test_df = pd.read_csv('../data/test.csv')

# Load metadata (feature descriptions and categories)
metadata_df = pd.read_csv('../data/metaData.csv')

# ============================================================================
# INITIAL DATA INSPECTION
# ============================================================================

# Check the shape of each dataset
# Shape returns (number of rows, number of columns)
print("📊 Dataset Shapes:")
print(f"  Training data: {train_df.shape[0]:,} rows × {train_df.shape[1]} columns")
print(f"  Test data: {test_df.shape[0]:,} rows × {test_df.shape[1]} columns")
print(f"  Metadata: {metadata_df.shape[0]} rows × {metadata_df.shape[1]} columns")

# Check memory usage
# Important for large datasets - helps us know if we need optimization
print(f"\n💾 Memory Usage:")
print(f"  Training: {train_df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print(f"  Test: {test_df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

# Quick sanity check
print(f"\n✅ Data loaded successfully!")
print(f"\n💡 Key Observation:")
print(f"   Train and test have the same number of features ({train_df.shape[1]} columns)")
print(f"   This is expected - test data should have same structure as training data")

## 2. Data Overview

### 🔍 First Look at the Data

Before any analysis, we should:
1. **Look at sample rows** to understand what the data looks like
2. **Check data types** to ensure columns are interpreted correctly
3. **Review metadata** to understand what each feature represents

This helps us catch issues early (e.g., numbers stored as text, missing values, etc.)

In [ ]:
# ============================================================================
# DISPLAY SAMPLE ROWS
# ============================================================================

# Display first few rows to get a sense of the data
# .head() shows the first 5 rows by default
print("\n=== First 5 rows of training data ===")
print("\n💡 What to look for:")
print("   - Are values in reasonable ranges?")
print("   - Do column names make sense?")
print("   - Any obvious data quality issues?\n")
display(train_df.head())

In [ ]:
# ============================================================================
# CHECK DATA TYPES AND MEMORY
# ============================================================================

# .info() provides a concise summary:
# - Column names
# - Non-null counts (helps identify missing values)
# - Data types (int64, float64, object, etc.)
# - Memory usage
print("\n=== Data Types and Memory Usage ===")
print("\n💡 What to check:")
print("   - Are numerical features stored as numbers (int64/float64)?")
print("   - Are there any 'object' types that should be numbers?")
print("   - Do we have the expected number of non-null values?\n")
train_df.info()

In [ ]:
# ============================================================================
# METADATA OVERVIEW
# ============================================================================

# The metadata file contains important information about each feature:
# - 'column': The feature name
# - 'type': Whether it's a feature, target, or identifier
# - 'category': Domain grouping (distance, growth, kinematics, etc.)
# - 'description': What the feature represents

print("\n=== Feature Metadata ===")
print("\n💡 This table helps us understand:")
print("   - What each feature measures")
print("   - How features are grouped by domain")
print("   - Which columns are features vs targets\n")
display(metadata_df)

In [ ]:
# ============================================================================
# FEATURE CATEGORIES
# ============================================================================

# Group features by their domain category
# This helps us understand the structure of our feature space
print("\n=== Feature Categories ===")
print("\n💡 Features are organized into domain-specific categories:")
print("   - Distance: How far the fire is from evacuation zones")
print("   - Growth: How the fire is expanding")
print("   - Kinematics: Fire movement patterns (speed, direction)")
print("   - Temporal: Time-related information\n")

# Filter for features only (exclude targets and identifiers)
feature_categories = metadata_df[metadata_df['type'] == 'feature'].groupby('category')['column'].apply(list)

for category, features in feature_categories.items():
    print(f"\n{category.upper()} ({len(features)} features):")
    for feat in features:
        print(f"  - {feat}")

## 3. Missing Values Analysis

### 🔍 Why Check for Missing Values?

Missing values can:
- **Break models**: Many algorithms can't handle NaN/null values
- **Indicate data quality issues**: Systematic missingness may reveal problems
- **Require special handling**: Different imputation strategies for different scenarios

### 📊 What We're Looking For:
1. **How many** values are missing per feature?
2. **What percentage** of data is missing?
3. **Are there patterns** in the missingness?

**Good News**: This dataset appears to be complete with no missing values!

In [ ]:
# ============================================================================
# CHECK FOR MISSING VALUES IN TRAINING DATA
# ============================================================================

print("\n=== Missing Values in Training Data ===")

# Count missing values per column
# .isnull() returns True for missing values, .sum() counts them
missing_train = train_df.isnull().sum()

# Calculate percentage of missing values
missing_train_pct = (missing_train / len(train_df)) * 100

# Create summary DataFrame
missing_summary = pd.DataFrame({
    'Missing_Count': missing_train,
    'Missing_Percentage': missing_train_pct
})

# Filter to show only columns with missing values
# Sort by count to see worst offenders first
missing_summary = missing_summary[missing_summary['Missing_Count'] > 0].sort_values('Missing_Count', ascending=False)

if len(missing_summary) > 0:
    print("\n⚠️ Features with missing values:")
    display(missing_summary)
    print("\n💡 Next steps would be:")
    print("   - Investigate WHY values are missing")
    print("   - Choose appropriate imputation strategy")
    print("   - Consider creating 'missing indicator' features")
else:
    print("✓ No missing values found in training data!")

# ============================================================================
# CHECK FOR MISSING VALUES IN TEST DATA
# ============================================================================

print("\n=== Missing Values in Test Data ===")

# Same process for test data
missing_test = test_df.isnull().sum()
missing_test_pct = (missing_test / len(test_df)) * 100

missing_summary_test = pd.DataFrame({
    'Missing_Count': missing_test,
    'Missing_Percentage': missing_test_pct
})
missing_summary_test = missing_summary_test[missing_summary_test['Missing_Count'] > 0].sort_values('Missing_Count', ascending=False)

if len(missing_summary_test) > 0:
    print("\n⚠️ Features with missing values:")
    display(missing_summary_test)
else:
    print("✓ No missing values found in test data!")

# Summary
print("\n💡 Key Takeaway:")
print("   Having no missing values is excellent! This means:")
print("   - No need for imputation strategies")
print("   - No risk of introducing bias through imputation")
print("   - Can focus on feature engineering and modeling")

## 4. Target Variable Analysis

### 🎯 Understanding Our Targets

This is a **survival analysis** problem, which means we have two target variables:

1. **`event`** (binary): Did the fire reach the evacuation zone?
   - 1 = Yes, fire reached within 72 hours (event occurred)
   - 0 = No, fire did NOT reach within 72 hours (censored)

2. **`time_to_hit_hours`** (continuous): When did it reach (if it did)?
   - For event=1: Actual time until fire reached zone
   - For event=0: Last observation time (72 hours)

### 📊 Why This Matters:

- **Event rate**: Shows how common the threat is
- **Censoring rate**: High censoring (~50%) confirms we need survival analysis methods
- **Time distribution**: Reveals when fires typically become threats

### ⚠️ Key Concept: Censored Data

**Censored observations** (event=0) are NOT failures or missing data!
- They tell us: "The fire hadn't reached the zone by 72 hours"
- We don't know if it would have reached at 80 hours, 100 hours, or never
- Standard classification models can't handle this properly
- Survival analysis models are specifically designed for this!

In [ ]:
# ============================================================================
# EVENT DISTRIBUTION (CENSORING ANALYSIS)
# ============================================================================

print("\n=== Event Distribution (Censoring Analysis) ===")

# Count how many fires reached evacuation zones (event=1) vs didn't (event=0)
event_counts = train_df['event'].value_counts()

# Calculate percentages
event_pct = train_df['event'].value_counts(normalize=True) * 100

# Create summary table
event_summary = pd.DataFrame({
    'Count': event_counts,
    'Percentage': event_pct
})
event_summary.index = ['Censored (0)', 'Event Occurred (1)']
display(event_summary)

print(f"\nCensoring Rate: {event_pct[0]:.2f}%")
print(f"Event Rate: {event_pct[1]:.2f}%")
print(f"\n⚠️  High censoring rate (~50%) confirms this is a survival analysis problem")

In [ ]:
# ============================================================================
# TIME TO HIT DISTRIBUTION
# ============================================================================

# Analyze the time_to_hit_hours variable
# This tells us WHEN fires reach evacuation zones
print("\n=== Time to Hit Hours - Summary Statistics ===")
print("\n💡 Understanding the output:")
print("   - count: Number of observations")
print("   - mean: Average time (influenced by censored observations at 72h)")
print("   - std: Spread of times")
print("   - min/max: Fastest and slowest times\n")
display(train_df['time_to_hit_hours'].describe())

# Compare time distributions by event status
# This is crucial for understanding censored vs actual event times
print("\n=== Time to Hit by Event Status ===")
print("\n💡 Key insight:")
print("   - Event=0 (censored): All values will be 72 hours (end of observation)")
print("   - Event=1 (occurred): Actual times when fires reached zones\n")
display(train_df.groupby('event')['time_to_hit_hours'].describe())

# Focus on actual events (event=1) for more meaningful statistics
events_only = train_df[train_df['event'] == 1]['time_to_hit_hours']
print(f"\n📊 For fires that HIT evacuation zones (event=1):")
print(f"   - Mean time: {events_only.mean():.2f} hours")
print(f"   - Median time: {events_only.median():.2f} hours (50% hit by this time)")
print(f"   - 25th percentile: {events_only.quantile(0.25):.2f} hours (25% hit by this time)")
print(f"   - 75th percentile: {events_only.quantile(0.75):.2f} hours (75% hit by this time)")

print("\n💡 Interpretation:")
print(f"   - Half of threatening fires reach zones within {events_only.median():.1f} hours")
print(f"   - This information helps prioritize early warning systems")

## 5. Numerical Features Analysis

### 🔢 Understanding Our Features

Now we'll analyze the numerical features that describe wildfire behavior:

**What we're looking for:**
1. **Feature count**: How many predictive features do we have?
2. **Value ranges**: Are features on similar scales? (important for some models)
3. **Variance**: Do features vary enough to be informative?
4. **Zero-inflation**: Are there many zeros? (may need special handling)

**Why this matters:**
- Features with no variance provide no information
- Features with many zeros may need transformation
- Understanding scales helps with feature engineering and model selection

In [ ]:
# ============================================================================
# IDENTIFY NUMERICAL FEATURES
# ============================================================================

# Get all numerical columns
# .select_dtypes(include=[np.number]) finds int64 and float64 columns
numerical_features = train_df.select_dtypes(include=[np.number]).columns.tolist()

# Remove non-feature columns (identifiers and targets)
# We only want predictive features for analysis
numerical_features = [f for f in numerical_features if f not in ['event_id', 'time_to_hit_hours', 'event']]

print(f"\n=== Numerical Features ({len(numerical_features)} total) ===")
print("\n💡 These are the features we'll use to predict wildfire threats\n")
print(numerical_features)

In [ ]:
# ============================================================================
# SUMMARY STATISTICS
# ============================================================================

# .describe() provides key statistics for each feature:
# - count: number of non-null values
# - mean: average value
# - std: standard deviation (measure of spread)
# - min/max: range of values
# - 25%/50%/75%: quartiles (distribution shape)

print("\n=== Summary Statistics for Numerical Features ===")
print("\n💡 What to look for:")
print("   - Large differences in scales (e.g., 0-1 vs 0-1000)")
print("   - Features with very small std (low variance)")
print("   - Unusual min/max values (potential outliers)\n")
display(train_df[numerical_features].describe().T)

In [ ]:
# ============================================================================
# CHECK FOR LOW VARIANCE FEATURES
# ============================================================================

# Features with zero variance (constant values) provide no information
# They should be removed before modeling

print("\n=== Features with Low Variance ===")
print("\n💡 Why this matters:")
print("   - Constant features (std=0) can't help predict anything")
print("   - They waste computational resources")
print("   - Should be removed before modeling\n")

low_variance_features = []

for col in numerical_features:
    # Check if standard deviation is zero (no variation)
    if train_df[col].std() == 0:
        low_variance_features.append(col)
        print(f"  ⚠️ {col}: constant value = {train_df[col].iloc[0]}")

if not low_variance_features:
    print("✓ No constant-value features found")
    print("  All features show variation - good for modeling!")

In [ ]:
# ============================================================================
# CHECK FOR ZERO-INFLATED FEATURES
# ============================================================================

# Zero-inflated features have many zero values
# This is common in wildfire data (e.g., no growth in certain directions)

print("\n=== Features with High Zero Percentage ===")
print("\n💡 What are zero-inflated features?")
print("   - Features where >50% of values are exactly zero")
print("   - Common in sparse data (e.g., fire not growing in all directions)")
print("   - Need special handling for modeling\n")

# Calculate percentage of zeros for each feature
zero_pct = (train_df[numerical_features] == 0).sum() / len(train_df) * 100

# Filter features with >50% zeros
high_zero_features = zero_pct[zero_pct > 50].sort_values(ascending=False)

if len(high_zero_features) > 0:
    print(f"⚠️ Found {len(high_zero_features)} zero-inflated features:")
    for feat, pct in high_zero_features.items():
        print(f"  - {feat}: {pct:.2f}% zeros")
    
    print(f"\n💡 Recommended handling:")
    print("   1. Create binary indicator: is_zero (0/1)")
    print("   2. Transform non-zero values: log1p(value)")
    print("   3. This preserves information about both presence and magnitude")
    print("\n   Example: For feature 'growth_rate':")
    print("   - Original: [0, 0, 5, 0, 10] → has_growth: [0, 0, 1, 0, 1]")
    print("   - Transform: [0, 0, 5, 0, 10] → log_growth: [0, 0, 1.79, 0, 2.40]")
else:
    print("✓ No features with >50% zeros")
    print("  Standard scaling should work well for all features")

## 6. Feature Category Analysis

### 📂 Understanding Feature Groups

Features are organized into domain-specific categories:
- **Distance**: How far the fire is from evacuation zones
- **Growth**: How the fire perimeter is expanding
- **Kinematics**: Fire movement (speed, direction, acceleration)
- **Temporal**: Time-related information

**Why analyze by category?**
- Helps understand which aspects of fire behavior are most predictive
- Guides feature engineering (e.g., creating interaction features within categories)
- Identifies if certain categories need more features or better representation

In [ ]:
# ============================================================================
# ANALYZE FEATURES BY CATEGORY
# ============================================================================

# Look at statistics for each feature category
# This helps us understand which types of features might be most useful

print("\n=== Feature Statistics by Category ===")
print("\n💡 Comparing categories helps us:")
print("   - Identify which aspects of fire behavior vary most")
print("   - Spot categories that might need more features")
print("   - Guide feature engineering priorities\n")

for category, features in feature_categories.items():
    print(f"\n{'='*60}")
    print(f"{category.upper()} FEATURES")
    print(f"{'='*60}")
    
    # Filter features that exist in training data
    existing_features = [f for f in features if f in train_df.columns]
    
    if existing_features:
        # Show key statistics: mean, std (variation), min, max (range)
        stats = train_df[existing_features].describe().T[['mean', 'std', 'min', 'max']]
        display(stats)
    else:
        print("No features found in training data")

## 7. Correlation Analysis

### 🔗 Understanding Feature Relationships

Correlation analysis reveals:
1. **Which features predict the target** (high correlation with target)
2. **Which features are redundant** (high correlation with each other)
3. **Potential multicollinearity issues** (can hurt some models)

### 📊 Correlation Coefficient (r):
- **r = 1**: Perfect positive correlation (as X increases, Y increases)
- **r = 0**: No linear relationship
- **r = -1**: Perfect negative correlation (as X increases, Y decreases)
- **|r| > 0.7**: Strong correlation
- **|r| > 0.9**: Very strong (potential multicollinearity)

### ⚠️ Important Notes:
- Correlation only measures LINEAR relationships
- High correlation between features (multicollinearity) can be problematic for some models
- Low correlation doesn't mean a feature is useless (could have non-linear relationship)

In [ ]:
# ============================================================================
# CORRELATION WITH TARGET VARIABLES
# ============================================================================

# Find features most correlated with our targets
# These are likely to be the most predictive features

print("\n=== Top 15 Features Correlated with Time to Hit ===")
print("\n💡 Interpretation:")
print("   - Positive correlation: Higher feature value → longer time to hit")
print("   - Negative correlation: Higher feature value → shorter time to hit")
print("   - We use absolute value to rank by strength\n")

# Calculate correlations with time_to_hit_hours
time_corr = train_df[numerical_features + ['time_to_hit_hours']].corr()['time_to_hit_hours'].drop('time_to_hit_hours')

# Sort by absolute value (strength) and take top 15
time_corr_sorted = time_corr.abs().sort_values(ascending=False).head(15)
display(pd.DataFrame(time_corr[time_corr_sorted.index]))

print("\n=== Top 15 Features Correlated with Event ===")
print("\n💡 Interpretation:")
print("   - Positive correlation: Higher feature value → more likely to hit")
print("   - Negative correlation: Higher feature value → less likely to hit\n")

# Calculate correlations with event
event_corr = train_df[numerical_features + ['event']].corr()['event'].drop('event')

# Sort by absolute value and take top 15
event_corr_sorted = event_corr.abs().sort_values(ascending=False).head(15)
display(pd.DataFrame(event_corr[event_corr_sorted.index]))

In [ ]:
# ============================================================================
# MULTICOLLINEARITY DETECTION
# ============================================================================

# Find pairs of features that are highly correlated with each other
# High multicollinearity can cause problems for some models

print("\n=== Highly Correlated Feature Pairs (|r| > 0.9) ===")
print("\n💡 Why this matters:")
print("   - Highly correlated features provide redundant information")
print("   - Can cause instability in linear models (coefficients flip signs)")
print("   - May need to remove one feature from each pair")
print("   - Tree-based models handle this better\n")

# Calculate correlation matrix for all features
corr_matrix = train_df[numerical_features].corr()

# Find highly correlated pairs (|r| > 0.9)
# We only check upper triangle to avoid duplicates
high_corr_pairs = []
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        if abs(corr_matrix.iloc[i, j]) > 0.9:
            high_corr_pairs.append({
                'Feature 1': corr_matrix.columns[i],
                'Feature 2': corr_matrix.columns[j],
                'Correlation': corr_matrix.iloc[i, j]
            })

if high_corr_pairs:
    high_corr_df = pd.DataFrame(high_corr_pairs).sort_values('Correlation', ascending=False, key=abs)
    display(high_corr_df)
    print(f"\n⚠️ Found {len(high_corr_pairs)} highly correlated pairs")
    print(f"\n💡 Recommended actions:")
    print("   1. For linear models: Remove one feature from each pair")
    print("   2. For tree models: Can keep both (they handle multicollinearity)")
    print("   3. Consider PCA or other dimensionality reduction")
else:
    print("✓ No feature pairs with |correlation| > 0.9")
    print("  Low multicollinearity - good for all model types!")

## 8. Temporal Features Analysis

### ⏰ Time-Based Patterns

Temporal features capture when fires start:
- **Hour of day**: Are fires more dangerous at certain times?
- **Day of week**: Weekend vs weekday patterns?
- **Month**: Seasonal effects (dry season, wind patterns)?

**Why analyze temporal patterns?**
- May reveal systematic patterns (e.g., afternoon fires spread faster)
- Helps understand if time-based features are predictive
- Guides whether to create additional temporal features

**Note**: These are metadata about when the observation started, not about fire behavior itself.

In [ ]:
# ============================================================================
# TEMPORAL METADATA DISTRIBUTION
# ============================================================================

# Analyze when fire observations started
# This helps identify if there are temporal patterns

print("\n=== Event Start Hour Distribution ===")
print("\n💡 What to look for:")
print("   - Are observations evenly distributed across hours?")
print("   - Are certain hours over/under-represented?")
print("   - Could indicate data collection patterns\n")
display(train_df['event_start_hour'].value_counts().sort_index())

print("\n=== Event Start Day of Week Distribution ===")
print("\n💡 Checking for:")
print("   - Weekend vs weekday differences")
print("   - Potential data collection biases\n")
day_names = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
dow_counts = train_df['event_start_dayofweek'].value_counts().sort_index()
for idx, count in dow_counts.items():
    print(f"{day_names[idx]}: {count}")

print("\n=== Event Start Month Distribution ===")
print("\n💡 Looking for:")
print("   - Seasonal patterns (fire season vs off-season)")
print("   - Data collection coverage across the year\n")
month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
month_counts = train_df['event_start_month'].value_counts().sort_index()
for idx, count in month_counts.items():
    print(f"{month_names[idx-1]}: {count}")

print("\n💡 Interpretation:")
print("   - Uniform distribution: Temporal features may not be very predictive")
print("   - Skewed distribution: Temporal patterns might be important")
print("   - Consider creating cyclical features (sin/cos) for hour and month")

## 9. Outlier Detection

### 🔍 Understanding Outliers

**What are outliers?**
- Data points that are significantly different from other observations
- Can be errors OR legitimate extreme values

**Why detect them?**
- **Data quality**: Identify potential errors or measurement issues
- **Model impact**: Outliers can heavily influence some models (e.g., linear regression)
- **Domain insights**: Extreme cases might reveal important patterns

### 📊 IQR Method (Interquartile Range):

**How it works:**
1. Calculate Q1 (25th percentile) and Q3 (75th percentile)
2. Calculate IQR = Q3 - Q1 (middle 50% of data)
3. Define outliers as values outside: [Q1 - 1.5×IQR, Q3 + 1.5×IQR]

**Why 1.5×IQR?**
- Standard threshold from Tukey's method
- Captures ~99.3% of data if normally distributed
- Can adjust threshold: 1.5 (standard), 3.0 (extreme outliers only)

### ⚠️ Important for Wildfire Data:
- Outliers may represent extreme but VALID wildfire events
- Don't automatically remove them - investigate first!
- Extreme fires might be the most important to predict

In [ ]:
# ============================================================================
# OUTLIER DETECTION FUNCTION
# ============================================================================

def detect_outliers_iqr(df, features, threshold=1.5):
    """
    Detect outliers using Interquartile Range (IQR) method.
    
    Parameters:
    -----------
    df : pd.DataFrame
        DataFrame containing the features
    features : list
        List of feature names to check for outliers
    threshold : float, default=1.5
        IQR multiplier for outlier detection
        - 1.5: Standard (Tukey's method)
        - 3.0: Only extreme outliers
    
    Returns:
    --------
    pd.DataFrame
        Summary of outliers per feature
    """
    outlier_summary = []
    
    for col in features:
        # Calculate quartiles
        Q1 = df[col].quantile(0.25)  # 25th percentile
        Q3 = df[col].quantile(0.75)  # 75th percentile
        IQR = Q3 - Q1  # Interquartile range (middle 50%)
        
        # Define outlier boundaries
        lower_bound = Q1 - threshold * IQR
        upper_bound = Q3 + threshold * IQR
        
        # Count outliers (values outside boundaries)
        outliers = ((df[col] < lower_bound) | (df[col] > upper_bound)).sum()
        outlier_pct = (outliers / len(df)) * 100
        
        # Only record features that have outliers
        if outliers > 0:
            outlier_summary.append({
                'Feature': col,
                'Outlier_Count': outliers,
                'Outlier_Percentage': outlier_pct
            })
    
    return pd.DataFrame(outlier_summary).sort_values('Outlier_Percentage', ascending=False)

# ============================================================================
# RUN OUTLIER DETECTION
# ============================================================================

print("\n=== Top 20 Features with Most Outliers (IQR method) ===")
print("\n💡 Interpreting results:")
print("   - High outlier % doesn't mean bad data")
print("   - For wildfire data, outliers often represent extreme events")
print("   - These extreme cases might be the most important to predict!\n")

outlier_summary = detect_outliers_iqr(train_df, numerical_features)
display(outlier_summary.head(20))

print(f"\n💡 Next steps for outliers:")
print("   1. Investigate features with >30% outliers")
print("   2. Check if outliers are measurement errors or valid extremes")
print("   3. Consider robust scaling (less sensitive to outliers)")
print("   4. Tree-based models handle outliers well (no need to remove)")
print("\n⚠️ Important: Don't automatically remove outliers in wildfire data!")
print("   Extreme fires are often the most critical to predict correctly.")

## 10. Train vs Test Comparison

### 🔄 Checking for Data Shift

**What is data shift?**
- When test data has different statistical properties than training data
- Also called "distribution shift" or "dataset shift"

**Why does it matter?**
- Models learn patterns from training data
- If test data is different, model performance may degrade
- Large shifts can indicate:
  - Different time periods (temporal shift)
  - Different geographic regions
  - Data collection changes

**What we're checking:**
1. **Sample sizes**: Are train/test proportions reasonable?
2. **Feature means**: Do features have similar average values?
3. **Large differences**: Any features with >20% difference?

**Ideal scenario**: Train and test distributions are similar (no major shifts)

In [ ]:
# ============================================================================
# COMPARE TRAIN VS TEST DISTRIBUTIONS
# ============================================================================

print("\n=== Train vs Test - Basic Statistics Comparison ===")
print("\n💡 Checking for:")
print("   - Reasonable train/test split ratio")
print("   - Similar feature distributions")
print("   - Potential data shift issues\n")

print(f"Training samples: {len(train_df):,}")
print(f"Test samples: {len(test_df):,}")
print(f"Train/Test ratio: {len(train_df)/len(test_df):.2f}")

# Compare feature distributions
print("\n=== Feature Mean Comparison (Train vs Test) ===")
print("\n💡 What we're comparing:")
print("   - Mean values of each feature in train vs test")
print("   - Large differences might indicate distribution shift\n")

# Calculate means for both datasets
comparison = pd.DataFrame({
    'Train_Mean': train_df[numerical_features].mean(),
    'Test_Mean': test_df[numerical_features].mean()
})

# Calculate absolute and percentage differences
comparison['Difference'] = comparison['Train_Mean'] - comparison['Test_Mean']
comparison['Pct_Difference'] = (comparison['Difference'] / comparison['Train_Mean'].abs()) * 100

# Show features with largest differences
print("Top 10 features with largest percentage difference:")
display(comparison.sort_values('Pct_Difference', ascending=False, key=abs).head(10))

# Check for significant distribution shifts
# Using 20% as threshold for "significant" difference
large_shifts = comparison[comparison['Pct_Difference'].abs() > 20]

if len(large_shifts) > 0:
    print(f"\n⚠️ Found {len(large_shifts)} features with >20% difference between train and test")
    print("\n💡 Potential implications:")
    print("   - Model may not generalize well to test data")
    print("   - Consider investigating these features further")
    print("   - May need domain adaptation techniques")
else:
    print("\n✓ No major distribution shifts detected")
    print("  Train and test data appear to come from similar distributions")
    print("  This is good - model should generalize well!")

## 11. Key Insights Summary

### 📋 Pulling It All Together

This section summarizes the most important findings from our exploratory data analysis.
These insights will guide our feature engineering and modeling decisions.

In [ ]:
# ============================================================================
# COMPREHENSIVE EDA SUMMARY
# ============================================================================

print("\n" + "="*80)
print("🔥 KEY INSIGHTS FROM EXPLORATORY DATA ANALYSIS 🔥")
print("="*80)

# Dataset Overview
print(f"\n1️⃣ DATASET SIZE:")
print(f"   - Training samples: {len(train_df):,}")
print(f"   - Test samples: {len(test_df):,}")
print(f"   - Total features: {len(numerical_features)}")
print(f"   💡 Sufficient data for training robust models")

# Target Variable Analysis
print(f"\n2️⃣ TARGET VARIABLE (SURVIVAL ANALYSIS):")
print(f"   - Event rate: {event_pct[1]:.2f}% (fires that hit within 72h)")
print(f"   - Censoring rate: {event_pct[0]:.2f}% (fires that didn't hit)")
print(f"   - Mean time to hit: {train_df['time_to_hit_hours'].mean():.2f} hours")
print(f"   - Median time to hit: {train_df['time_to_hit_hours'].median():.2f} hours")
print(f"   ⚠️  High censoring rate (~50%) confirms this is a survival analysis problem")
print(f"   💡 Cannot use standard classification - need survival analysis methods")

# Data Quality Assessment
print(f"\n3️⃣ DATA QUALITY:")
print(f"   - Missing values: {'None ✓' if len(missing_summary) == 0 else 'Present ⚠️'}")
print(f"   - Features with >50% zeros: {len(high_zero_features)}")
print(f"   - Highly correlated pairs (|r|>0.9): {len(high_corr_pairs)}")
if len(missing_summary) == 0:
    print(f"   ✓ Complete dataset - no missing value imputation needed")
if len(high_zero_features) > 0:
    print(f"   ⚠️ Zero-inflated features need special handling")
if len(high_corr_pairs) > 0:
    print(f"   ⚠️ Multicollinearity detected - may need feature selection")

# Feature Organization
print(f"\n4️⃣ FEATURE CATEGORIES:")
for category, features in feature_categories.items():
    print(f"   - {category.capitalize()}: {len(features)} features")
print(f"   💡 Features organized by wildfire behavior aspects")

# Modeling Strategy
print(f"\n5️⃣ MODELING IMPLICATIONS:")
print(f"   - Problem type: Survival analysis with censored data")
print(f"   - Suitable models: Cox PH, Random Survival Forests, DeepSurv")
print(f"   - Evaluation: Hybrid Score = 0.3×C-index + 0.7×(1-Brier Score)")
print(f"   - Multi-horizon: Must predict at 12h, 24h, 48h, 72h")
print(f"   - Constraint: Probabilities must be monotonic (increasing over time)")
print(f"   💡 Specialized survival analysis tools required (e.g., scikit-survival)")

# Feature Engineering Roadmap
print(f"\n6️⃣ FEATURE ENGINEERING PRIORITIES:")
if len(high_zero_features) > 0:
    print(f"   1. Handle {len(high_zero_features)} zero-inflated features:")
    print(f"      - Create binary indicators (has_value)")
    print(f"      - Apply log1p transformation to non-zero values")
if len(high_corr_pairs) > 0:
    print(f"   2. Address {len(high_corr_pairs)} highly correlated pairs:")
    print(f"      - Remove redundant features OR")
    print(f"      - Use PCA/dimensionality reduction")
print(f"   3. Create domain-specific features:")
print(f"      - Threat velocity (distance/time)")
print(f"      - Risk scores (combining multiple factors)")
print(f"      - Directional features (fire heading toward zone?)")
print(f"   4. Generate interaction features:")
print(f"      - distance × growth_rate")
print(f"      - distance × speed")
print(f"      - growth × wind_direction")
print(f"   5. Apply robust scaling (handles outliers well)")

print("\n" + "="*80)
print("✅ EDA COMPLETE - Ready for Feature Engineering and Modeling!")
print("="*80)

## Next Steps

Based on this EDA, the following notebooks will address:

1. **Visualization Notebook (02)**: 
   - Create detailed visualizations of distributions, correlations, and survival curves
   - Kaplan-Meier survival estimates
   - Feature distribution comparisons

2. **Feature Engineering Notebook (03)**: 
   - Handle zero-inflated features (binary indicators + log1p transforms)
   - Create interaction features (distance × growth, distance × speed)
   - Engineer domain-specific wildfire features
   - Address multicollinearity through feature selection
   - Apply robust scaling

3. **Modeling Notebook (04)**: 
   - Implement survival analysis models (Cox PH, Random Survival Forests)
   - Generate multi-horizon probability forecasts (12h, 24h, 48h, 72h)
   - Evaluate using Hybrid Score (C-index + Weighted Brier Score)
   - Ensure monotonicity constraints
   - Analyze feature importance

### Key Findings to Address:

- ✅ **No missing values** - clean dataset ready for analysis
- ⚠️ **High censoring rate (~50%)** - requires survival analysis approach
- ⚠️ **Zero-inflated features** - need special handling with binary indicators
- ⚠️ **High multicollinearity** - requires feature selection or dimensionality reduction
- ℹ️ **Temporal patterns** - may provide weak but useful signals
- ℹ️ **Distance metrics** - appear to be strong predictors based on correlation analysis